In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# =========================
# 1. Imports
# =========================
import os
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

# =========================
# 2. LOAD DATASET
# =========================
base_path = "/kaggle/input/datasets/nikunjnawal009/lr-devignx"

df = None
for root, dirs, files in os.walk(base_path):
    for f in files:
        if f.endswith(".csv"):
            path = os.path.join(root, f)
            print("✅ Loaded:", path)
            df = pd.read_csv(path)
            break
    if df is not None:
        break

if df is None:
    raise Exception("❌ Dataset not found")

print("\n📊 Shape:", df.shape)

# =========================
# 3. PREPROCESS
# =========================
df = df[['code', 'label']].dropna()
df.columns = ['text', 'label']
df['label'] = df['label'].astype(int)

# =========================
# 4. TRAIN-VAL-TEST SPLIT
# =========================
X_train, X_temp, y_train, y_temp = train_test_split(
    df["text"].astype(str),
    df["label"],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

print(f"Train size: {len(X_train)} | Val size: {len(X_val)} | Test size: {len(X_test)}")

# =========================
# 5. TF-IDF (CODE-AWARE ENHANCEMENT)
# =========================
print("\nVectorizing data...")
vectorizer = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.90,
    # 🔥 UPGRADE: This regex captures alphanumeric words AND punctuation/operators
    token_pattern=r'[a-zA-Z0-9_]+|[^\w\s]', 
    dtype=np.float32 
)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec   = vectorizer.transform(X_val)
X_test_vec  = vectorizer.transform(X_test)

# =========================
# 6. LOGISTIC REGRESSION MODEL
# =========================
print("\nTraining Logistic Regression...")
lr = LogisticRegression(
    max_iter=2000,
    C=2.0, # 🔥 UPGRADE: Allows the model to fit harder to the complex TF-IDF code features
    n_jobs=-1,
    random_state=42 
    # Removed class_weight="balanced" so the probabilities aren't artificially skewed
)

lr.fit(X_train_vec, y_train)

# =========================
# 7. MACRO-F1 THRESHOLD TUNING (ON VALIDATION SET)
# =========================
val_probs = lr.predict_proba(X_val_vec)[:, 1]

best_macro_f1 = 0
best_t = 0.5

print("\n🔍 Threshold tuning on Validation Set:")

# 🔥 UPGRADE: Finer steps (0.02) and a wider net
for t in np.arange(0.30, 0.70, 0.02):
    preds = (val_probs > t).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(y_val, preds, labels=[0, 1]).ravel()
    
    # Calculate Class 0 metrics
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision_0 = tn / (tn + fn) if (tn + fn) > 0 else 0
    f1_0 = 2 * (precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0

    # Calculate Class 1 metrics
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision_1 = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1_1 = 2 * (precision_1 * recall_1) / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0

    # 🔥 UPGRADE: Target Macro F1 to force the model to respect both classes equally
    macro_f1 = (f1_0 + f1_1) / 2

    print(f"t={t:.2f} → Macro_F1={macro_f1:.4f} | R0={recall_0:.2f}, R1={recall_1:.2f}")

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_t = t

print(f"\nBest threshold found: {best_t:.2f}")

# =========================
# 8. FINAL EVALUATION (ON TEST SET)
# =========================
print("\nEvaluating on Test Set...")
test_probs = lr.predict_proba(X_test_vec)[:, 1]
final_preds = (test_probs > best_t).astype(int)

acc = accuracy_score(y_test, final_preds)
report = classification_report(y_test, final_preds, output_dict=True)

print("\n✅ Accuracy:", acc)
print("\n📊 Classification Report:\n", classification_report(y_test, final_preds))

# =========================
# 9. SAVE RESULTS
# =========================
label_key = "1" if "1" in report else[k for k in report.keys() if k.startswith("1")][0]

results = {
    "Model": "LogisticRegression",
    "Dataset": "Devign",
    "Accuracy": acc,
    "Precision_vuln": report[label_key]['precision'],
    "Recall_vuln": report[label_key]['recall'],
    "F1_vuln": report[label_key]['f1-score'],
    "Macro_F1": report['macro avg']['f1-score'],
    "Best_threshold": best_t
}

pd.DataFrame([results]).to_csv("/kaggle/working/lr_devign_results.csv", index=False)

print("\n✅ Results saved in /kaggle/working/")

✅ Loaded: /kaggle/input/datasets/nikunjnawal009/lr-devignx/Devignx_validation.csv

📊 Shape: (2732, 2)
Train size: 2185 | Val size: 273 | Test size: 274

Vectorizing data...

Training Logistic Regression...

🔍 Threshold tuning on Validation Set:
t=0.30 → Macro_F1=0.4667 | R0=0.20, R1=0.91
t=0.32 → Macro_F1=0.4893 | R0=0.25, R1=0.86
t=0.34 → Macro_F1=0.5142 | R0=0.31, R1=0.82
t=0.36 → Macro_F1=0.5212 | R0=0.32, R1=0.81
t=0.38 → Macro_F1=0.5312 | R0=0.40, R1=0.71
t=0.40 → Macro_F1=0.5415 | R0=0.45, R1=0.66
t=0.42 → Macro_F1=0.5456 | R0=0.50, R1=0.61
t=0.44 → Macro_F1=0.5332 | R0=0.57, R1=0.50
t=0.46 → Macro_F1=0.5299 | R0=0.62, R1=0.44
t=0.48 → Macro_F1=0.5251 | R0=0.67, R1=0.39
t=0.50 → Macro_F1=0.5166 | R0=0.73, R1=0.32
t=0.52 → Macro_F1=0.5189 | R0=0.79, R1=0.29
t=0.54 → Macro_F1=0.4875 | R0=0.83, R1=0.21
t=0.56 → Macro_F1=0.4615 | R0=0.88, R1=0.15
t=0.58 → Macro_F1=0.4577 | R0=0.90, R1=0.13
t=0.60 → Macro_F1=0.4311 | R0=0.93, R1=0.09
t=0.62 → Macro_F1=0.4223 | R0=0.95, R1=0.08
t=0.64 